# Running Summary Statistics

Mean, variance, extrema, skewness and kurtosis of a stream in one pass with [OnlineStats.jl](https://github.com/joshday/OnlineStats.jl), under the three kinds of memory of the **Running Summary** tab: incremental, rolling window and exponentially weighted.

In [ ]:
Pollis.packages("OnlineStats", "Plots")

In [ ]:
using OnlineStats, Random, Statistics, Plots

Random.seed!(1)  # simulated: a stream of 10,000 draws from Exponential(1), whose mean and variance are 1 and median log(2) = 0.693
data = randexp(10_000)  # Replace data with your own stream of observations.

## Incremental Summary

A `Series` passes each observation to every statistic it holds, so the stream is read once. With the default equal weights, every observation so far counts the same.

In [ ]:
s = fit!(Series(Mean(), Variance(), Extrema(), Moments()), data)

println("Mean:     ", value(s[1]))
println("Variance: ", value(s[2]))
println("Std dev:  ", sqrt(value(s[2])))  # the square root of the variance
println("Extrema:  ", extrema(s[3]))
println("Skewness: ", skewness(s[4]))  # 2 for Exponential(1)
println("Kurtosis: ", kurtosis(s[4]))  # excess kurtosis: 6 for Exponential(1)

## Checking Against Batch Values

With equal weights the online mean and variance are exact: they differ from the batch values of `Statistics` only by rounding.

In [ ]:
println("Mean difference:     ", value(s[1]) - mean(data))
println("Variance difference: ", value(s[2]) - var(data))

## Adding New Observations

New observations update the same object. The earlier observations are not read again.

In [ ]:
more = randexp(1_000)  # 1,000 new observations
fit!(s, more)
println("Observations: ", nobs(s))
println("Mean:         ", value(s[1]))

## Rolling Window

`MovingWindow` keeps the last observations, here 100, and the summary is computed from them.

In [ ]:
window = fit!(MovingWindow(100, Float64), data)  # the last 100 observations
r = fit!(Series(Mean(), Variance(), Extrema()), value(window))

println("Mean of the window:     ", value(r[1]))
println("Variance of the window: ", value(r[2]))
println("Extrema of the window:  ", extrema(r[3]))

## Exponentially Weighted

`ExponentialWeight(100)` gives each new observation the weight 2/(100 + 1), so older observations count less and less. `Extrema` takes no weight.

In [ ]:
w = ExponentialWeight(100)
e = fit!(Series(Mean(weight=w), Variance(weight=w)), data)

println("Exponentially weighted mean:     ", value(e[1]))
println("Exponentially weighted variance: ", value(e[2]))

## A Stream Whose Level Shifts

Halfway through, the level rises by 2. The incremental mean mixes both halves, while the rolling and exponentially weighted means move to the new level. The exponentially weighted mean also starts below the others, because `ExponentialWeight` gives the first observation a weight below 1.

In [ ]:
shifted = vcat(randexp(5_000), randexp(5_000) .+ 2)  # the level rises from 1 to 3 halfway

incremental = Mean()
rolling = MovingWindow(100, Float64)
weighted = Mean(weight=ExponentialWeight(100))

mi = Float64[]; mr = Float64[]; me = Float64[]
for x in shifted
    fit!(incremental, x); fit!(rolling, x); fit!(weighted, x)
    push!(mi, value(incremental)); push!(mr, mean(value(rolling))); push!(me, value(weighted))
end
plot([mi mr me], label=["Incremental" "Rolling Window" "Exponentially Weighted"], xlabel="Observations", ylabel="Mean")

## Merging Two Partitions

Statistics built on separate parts of a stream, for example on two processors, combine with `merge!`. With equal weights the merged mean and variance equal those of the whole stream.

In [ ]:
a = fit!(Series(Mean(), Variance()), data[1:5_000])
b = fit!(Series(Mean(), Variance()), data[5_001:end])
merge!(a, b)

println("Merged mean:     ", value(a[1]), "   batch: ", mean(data))
println("Merged variance: ", value(a[2]), "   batch: ", var(data))